# Data Cleaning
ทำความสะอาดข้อมูลจาก 3 ไฟล์หลัก แล้ว export เป็น `cleaned/` folder:
- `master_results_cleaned.csv`
- `master_summary_cleaned.csv`
- `coords_cleaned.csv` (ใช้ output_with_coordinates แทน unit_position / unit_village)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('../')
OUT  = Path('cleaned/')
OUT.mkdir(exist_ok=True)

results  = pd.read_csv(BASE / 'master_results.csv')
summary  = pd.read_csv(BASE / 'master_summary.csv', index_col=0)
coords   = pd.read_csv(BASE / 'output_with_coordinates.csv', index_col=0)

print('Loaded:')
print(f'  results : {results.shape}')
print(f'  summary : {summary.shape}')
print(f'  coords  : {coords.shape}')

---
## 1. Clean master_results.csv

### 1.1 แก้ค่า type ผิด: 'บч' → 'บช'

In [ ]:
r = results.copy()

before = (r['type'] == 'บч').sum()
r['type'] = r['type'].replace('บч', 'บช')
print(f'แก้ type บч → บช: {before} rows')
print('type unique หลังแก้:', r['type'].unique())

### 1.2 กรองเฉพาะ province == 'ลำปาง'

In [ ]:
noise_provinces = r[r['province'] != 'ลำปาง']['province'].value_counts()
print('province noise ที่จะถูกลบ:')
print(noise_provinces)

before = len(r)
r = r[r['province'] == 'ลำปาง'].copy()
print(f'\nลบ noise province: {before} → {len(r)} rows ({before - len(r)} ลบออก)')

In [ ]:
### 1.2b แก้ชื่ออำเภอ (district) ที่ OCR เพี้ยน
DISTRICT_MAP = {
    'งาม':             'งาว',
    'องาว':            'งาว',
    'อำเภองาว':        'งาว',
    'อำเภอวังเหนือ':   'วังเหนือ',
    'อำเภอเมืองปาน':   'เมืองปาน',
    'อำเภอเมืองลำปาง': 'เมืองลำปาง',
    'นางลิ้นจี่':      'แจ้ห่ม',   # OCR เพี้ยน ตำบลนางลิ้นจี่อยู่ในอำเภอแจ้ห่ม
}

before_d = r['district'].nunique()
r['district'] = r['district'].replace(DISTRICT_MAP)
after_d = r['district'].nunique()
print(f'district unique: {before_d} → {after_d}')
print('district ทั้งหมด:', sorted(r['district'].unique()))

### 1.3 แปลง score เป็นตัวเลข และลบแถวผิดปกติ

In [ ]:
r['score'] = pd.to_numeric(r['score'], errors='coerce')

# score = -1 → missing value (เหมือน pattern ใน summary)
neg_one = r[r['score'] == -1]
print(f'score == -1 (missing): {len(neg_one)} rows')
print(neg_one['type'].value_counts())

# score < 0 แต่ไม่ใช่ -1
other_neg = r[(r['score'] < 0) & (r['score'] != -1)]
print(f'\nscore < 0 อื่นๆ: {len(other_neg)} rows')
if len(other_neg) > 0:
    print(other_neg[['district','sub-district','unit_number','name','score']])

In [ ]:
# score > 99999 → OCR ต่อเลขกัน
bad_high = r[r['score'] > 99999]
print(f'score > 99999 (OCR ต่อเลข): {len(bad_high)} rows')
print(bad_high[['district','sub-district','unit_number','name','score']])

# score มีทศนิยม (OCR อ่านเลขผิด เช่น 72.80, 0.50)
bad_frac = r[(r['score'] > 0) & (r['score'] % 1 != 0) & (r['unit_number'] != -1)]
print(f'\nscore มีทศนิยมผิดปกติ: {len(bad_frac)} rows')
print(bad_frac[['district','sub-district','unit_number','name','score']])

In [ ]:
before = len(r)

# ลบ: score < 0 (รวม -1) และ score > 99999
r = r[(r['score'] >= 0) & (r['score'] <= 99999)].copy()

# ปัดทศนิยมที่เกิดจาก OCR → round เป็น int
r['score'] = r['score'].round().astype('Int64')

print(f'ลบแถวผิดปกติ: {before} → {len(r)} rows (ลบ {before - len(r)} rows)')
print('score dtype:', r['score'].dtype)
print('score min:', r['score'].min(), '| max:', r['score'].max())

### 1.4 Normalize ชื่อผู้สมัคร ส.ส. เขต (Entity Resolution)

In [ ]:
# ผู้สมัครจริงมี 7 คน ที่เหลือเป็น OCR เพี้ยน
CANDIDATE_MAP = {
    # นายดาซัย เอกปฐพี
    'นายดาชัย เอกปฐพี':   'นายดาซัย เอกปฐพี',
    'นายตาซัย เอกปฐพี':   'นายดาซัย เอกปฐพี',
    'นายดาซัย เอกปฏิพี':  'นายดาซัย เอกปฐพี',
    'นายดาชาย เอกปฐพี':   'นายดาซัย เอกปฐพี',
    'นายตาชัย เอกปฐพี':   'นายดาซัย เอกปฐพี',
    'นายดำชัย เอกปฐพี':   'นายดาซัย เอกปฐพี',
    'นายดาชัย เอกปฏิพี':  'นายดาซัย เอกปฐพี',
    'นายตาซัย เอกปฏิพี':  'นายดาซัย เอกปฐพี',
    'นายดาซัย เออปฐพี':   'นายดาซัย เอกปฐพี',
    'นายดาซัย เอกปรูพี':  'นายดาซัย เอกปฐพี',

    # นางสาวสุวิภา กุศลจูง
    'นางสาวสุวิภา กุศลจุง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภิภา กุศลจูง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภิภา กุศลจุง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวิภา กุศลงู':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวิภา กุศลถุง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวิภา กุศลอุง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวิภา กุคลุง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวิภา กูคลุง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวิภา กฤตจูง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุวภา กุศลจูง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภาวิภา กุศลจูง':  'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภาว กุศลจูง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภา กุศลจูง':      'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภา กุศลจุง':      'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภา กุศลจรุง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภิวา กุศลจูง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภภา กุศลจูง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภารา กุศลจุง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุชิภา กุศลจูง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุรภา กุศลจุง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุริภา กุศลจูง':    'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุริภา กุศลจุง':    'นางสาวสุวิภา กุศลจูง',
    'นายสารสุภา กุคลจรุง':     'นางสาวสุวิภา กุศลจูง',
    'นางสาวสุภาวิกฤตจูง':      'นางสาวสุวิภา กุศลจูง',

    # นายธนาธร โล่ห์สุนทร
    'นายธนาธร โล่หลุนทร':   'นายธนาธร โล่ห์สุนทร',
    'นายธนาธร โล่หลุ่นทร':  'นายธนาธร โล่ห์สุนทร',
    'นายธนาธร ไล่ห์สุนทร':  'นายธนาธร โล่ห์สุนทร',
    'นายธนาธร โสฬสุนทร':    'นายธนาธร โล่ห์สุนทร',
    'นายธนาธร โลห์สุนทร':   'นายธนาธร โล่ห์สุนทร',
    'นายธนาธร โล่ห์สนทร':   'นายธนาธร โล่ห์สุนทร',
    'นายธนาธร ไล่หุ่นทร':   'นายธนาธร โล่ห์สุนทร',

    # นางสาววิชุดา ว่องวัฒนวิโรจน์
    'นางสาววิชุดา ว่องวัฒน์โรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวัฒนวโรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวัฒนาโรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวัฒนาวิโรจน์':   'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวัฒน์วโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวัฒน์วิโรจน์':   'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวันวนิโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่องวันณวโรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา ว่วงวัฒน์วโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชุดา อ่วงวัฒนวิโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชชา ว่องวัฒนวิโรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชชา ว่องวัฒนวโรจน์':      'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชิตา ว่องวัฒนวิโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชาดา ว่องวัฒนวิโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชชาดา ว่องวัฒนวิโรจน์':   'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิชชาดา ว่องวัฒนาโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิฑูชา ว่องวัฒนวิโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิฑูดา ว่องวัฒน์โรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิฑูดา ว่องวัฒน์วิโรจน์':   'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิฑูดา ว่องวัฒนวิโรจน์':    'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาวรวิชุดา ว่องวัฒนวิโรจน์':   'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาวรวิชดา ว่องวัฒนวโรจน์':     'นางสาววิชุดา ว่องวัฒนวิโรจน์',
    'นางสาววิฤดา วงศ์พรมไพรณ์':        'นางสาววิชุดา ว่องวัฒนวิโรจน์',

    # พันเอกสันทัด ภัทรกิตตินนท์
    'พันเอกสันทัด ภัทรกิตตินันท์':  'พันเอกสันทัด ภัทรกิตตินนท์',
    'พันเอกสันทัด ภักรกิตตินนท์':   'พันเอกสันทัด ภัทรกิตตินนท์',
    'พันเอกสันทัด ภัทรคิตตินนท์':   'พันเอกสันทัด ภัทรกิตตินนท์',
    'พันเอกสันทัด ภัทรเกิดตินนท์':  'พันเอกสันทัด ภัทรกิตตินนท์',
    'พันเอกสืบทัต ภัทรเกิดสินท์':   'พันเอกสันทัด ภัทรกิตตินนท์',

    # นายสมบูรณ์ รูปสะอาด
    'นายสมบูรณ์ รูปละอาด': 'นายสมบูรณ์ รูปสะอาด',

    # นายศรีพรหม หอมยก
    'นายศรีพรหมอ หอมยก':  'นายศรีพรหม หอมยก',
    'นายศรีพรหมอ มอมยก':  'นายศรีพรหม หอมยก',
    'นายศรีพรหมอ มอยยก':  'นายศรีพรหม หอมยก',
    'นายศรีพรรณ หอมเอก':  'นายศรีพรหม หอมยก',
    'นายศรีพรหมอภย':       'นายศรีพรหม หอมยก',
}

mask_ket = r['type'] == 'เขต'
before_ket = r[mask_ket]['name'].nunique()
r.loc[mask_ket, 'name'] = r.loc[mask_ket, 'name'].replace(CANDIDATE_MAP)
after_ket = r[mask_ket]['name'].nunique()
print(f'เขต unique name: {before_ket} → {after_ket}')
print('ผู้สมัครที่เหลือ:', sorted(r[mask_ket]['name'].unique()))

In [ ]:
# Dictionary mapping ชื่อ OCR เพี้ยน → ชื่อมาตรฐาน
PARTY_MAP = {
    # ก้าวไกล / วิชชั่นใหม่ (Vision New)
    'ก้าวลิสระ': 'ก้าวไกล', 'ก้าวลิกระยะ': 'ก้าวไกล', 'ก้าวลิตระ': 'ก้าวไกล',
    'ก้าวลิตรระ': 'ก้าวไกล', 'ก้าวลิตรสะ': 'ก้าวไกล', 'ก้าวลิตราะ': 'ก้าวไกล',
    'ก้าวลิละระ': 'ก้าวไกล', 'ก้าวลิสง': 'ก้าวไกล', 'ก้าวลิสงระ': 'ก้าวไกล',
    'ก้าวลิสรละ': 'ก้าวไกล', 'ก้าวลิสรสะ': 'ก้าวไกล', 'ก้าวลิ่วสระ': 'ก้าวไกล',
    'ก้าวล่วง': 'ก้าวไกล', 'ก้าวอิสระ': 'ก้าวไกล', 'ก้าวก้มลง': 'ก้าวไกล',
    'ไทยก้าวใหม่': 'ก้าวไกล', 'ไทยกว้าใหม่': 'ก้าวไกล', 'ไทยกว่าใหม่': 'ก้าวไกล',
    'ไทยกว้างใหม่': 'ก้าวไกล', 'ไทยกว้าไหม': 'ก้าวไกล', 'ไทยกว้าใหญ่': 'ก้าวไกล',
    'ไทยกว้างใหญ่': 'ก้าวไกล', 'ไทยกว้านใหม่': 'ก้าวไกล', 'ไทยกว้าวใหม่': 'ก้าวไกล',
    'ไทยกาวใหม่': 'ก้าวไกล',

    # วิชชั่นใหม่
    'วิชช์นใหม่': 'วิชชั่นใหม่', 'วิชชินใหม่': 'วิชชั่นใหม่', 'วิชชิมใหม่': 'วิชชั่นใหม่',
    'วิชชิ่นใหม่': 'วิชชั่นใหม่', 'วิชช์ขึ้นใหม่': 'วิชชั่นใหม่', 'วิชช์ชั่นใหม่': 'วิชชั่นใหม่',
    'วิชช์ใหม่': 'วิชชั่นใหม่', 'วิชชันใหม่': 'วิชชั่นใหม่', 'วิชัยขับใหม่': 'วิชชั่นใหม่',
    'วิชัยขั้นใหม่': 'วิชชั่นใหม่', 'วิชัยขึ้นใหม่': 'วิชชั่นใหม่', 'วิชัยชั้นใหม่': 'วิชชั่นใหม่',
    'วิชัยซันใหม่': 'วิชชั่นใหม่', 'วิชัยใหม่': 'วิชชั่นใหม่', 'วิชาชีพใหม่': 'วิชชั่นใหม่',
    'วิชัยขับใหญ่': 'วิชชั่นใหม่', 'ภิติใหม่': 'วิชชั่นใหม่',

    # สร้างอนาคตไทย
    'สร้างอนาคคไทย': 'สร้างอนาคตไทย', 'สรางอนาคตไทย': 'สร้างอนาคตไทย',
    'สรางอนาคไทย': 'สร้างอนาคตไทย', 'สร้างอนาคตกไทย': 'สร้างอนาคตไทย',
    'สร้างอนาคไทย': 'สร้างอนาคตไทย',

    # ไทยก้าวหน้า
    'ไทยก้าวหน้าแห่งประเทศไทย': 'ไทยก้าวหน้า',
    'ไทยก้าวหน้า นาแห่งประเทศไทย': 'ไทยก้าวหน้า',
    'ไทยก้าวหน้า นำแนวประเทศไทย': 'ไทยก้าวหน้า',
    'ไทยก้าวหน้า นำแห่งประเทศไทย': 'ไทยก้าวหน้า',
    'ไทยก้าวหน้า มาแห่งประเทศไทย': 'ไทยก้าวหน้า',
    'ไทยก้าวหน้า แห่งประเทศไทย': 'ไทยก้าวหน้า',
    'ไทยก้าวหน้านานดีประเทศไทย': 'ไทยก้าวหน้า',

    # ไทยชนะ
    'ไทยชนะ รวม': 'ไทยชนะ', 'ไทยชนะรวม': 'ไทยชนะ', 'ไทมชนะ': 'ไทยชนะ',

    # ไทยภักดี
    'ไทยภักดี ธรรม': 'ไทยภักดี',

    # ไทยพร้อม
    'ไทยพร้อม -': 'ไทยพร้อม',

    # ไทยพิทักษ์ธรรม
    'ไทยพิทักษ์กรรม': 'ไทยพิทักษ์ธรรม',

    # ไทยทรัพย์ทวี
    'ไทยทรัพย์ไว': 'ไทยทรัพย์ทวี', 'ไทยทรัพยวิถี': 'ไทยทรัพย์ทวี',
    'ไทยทรัพยากร': 'ไทยทรัพย์ทวี',

    # ไทยรวมไทย
    'ไทรวมพลัง': 'ไทยรวมไทย', 'ไทไรวมพลัง': 'ไทยรวมไทย',

    # ประชาธิปัตย์
    'ประชาธิปโดยใหม่': 'ประชาธิปัตย์', 'ประชาธิปไตยใหม่': 'ประชาธิปัตย์',
    'ประชาธิปัตย์ใหม่': 'ประชาธิปัตย์', 'ประชาธิปโดยใหม่': 'ประชาธิปัตย์',
    'ประชาชาติปัตย์': 'ประชาธิปัตย์', 'ใหม่ราชิติปัตย์': 'ประชาธิปัตย์',

    # ประชาไทย
    'ประชนไทย': 'ประชาไทย', 'ประชาอาสาชาติ': 'ประชาไทย',

    # ครูไทยเพื่อประชาชน
    'ครูไทยเทือประชาน': 'ครูไทยเพื่อประชาชน',

    # คลองไทย
    'คลองไทย สภาชาติ': 'คลองไทย', 'คลองไทยชาติ': 'คลองไทย',

    # เพื่อชาติไทย
    'เพื่อชาติใหม่': 'เพื่อชาติไทย', 'เพื่อชาติติไทย': 'เพื่อชาติไทย',
    'เพื่อชาติต่อไป': 'เพื่อชาติไทย',

    # เพื่อชีวิตใหม่
    'เพื่อขีวิตใหม่': 'เพื่อชีวิตใหม่', 'เทื่อชีวิตใหม่': 'เพื่อชีวิตใหม่',

    # เพื่อบ้านเมือง
    'เทือบ้านเมือง': 'เพื่อบ้านเมือง',

    # พลังไทยรักชาติ
    'พลังไทyrักชาติ': 'พลังไทยรักชาติ', 'พลังไทธรักชาติ': 'พลังไทยรักชาติ',
    'หลังไทยรักชาติ': 'พลังไทยรักชาติ',

    # พลวัต
    'พลวัด': 'พลวัต',

    # พิวจันทร์ (ถ้ามี)
    'พิวจัน': 'พิวจันทร์', 'พิวซัน': 'พิวจันทร์', 'พิวซันใหม่': 'พิวจันทร์',
    'พิวชัน': 'พิวจันทร์', 'พิวซิน': 'พิวจันทร์', 'พิวัฒน์': 'พิวจันทร์',
    'พิวขัน': 'พิวจันทร์', 'พิวจัน': 'พิวจันทร์', 'ฟิวชัน': 'พิวจันทร์',
    'ฟิวซัน': 'พิวจันทร์', 'พิกขัน': 'พิวจันทร์', 'พิกวัน': 'พิวจันทร์',
    'พิกุล': 'พิวจันทร์', 'พิกุลชน': 'พิวจันทร์', 'พิกุลัน': 'พิวจันทร์',
    'พิมพ์ขึ้น': 'พิวจันทร์', 'พิวจันทร์': 'พิวจันทร์', 'พิวซัน ใหม่': 'พิวจันทร์',
    'พืชขัน': 'พิวจันทร์', 'พืชชน': 'พิวจันทร์', 'พืชชัน': 'พิวจันทร์',
    'พืชซัน': 'พิวจันทร์', 'ฟิลิปปิน': 'พิวจันทร์',

    # รักชาติ
    'วักชาติ': 'รักชาติ',

    # รวมไทยสร้างชาติ
    'รวมใจไทย': 'รวมไทยสร้างชาติ',

    # ท้องที่ไทย
    'ห้องที่ไทย': 'ท้องที่ไทย', 'พ้องที่ไทย': 'ท้องที่ไทย',
    'ท้องที่ไทยใหม่': 'ท้องที่ไทย',

    # สังคมประชาธิปไตยไทย
    'สังคมประชาธิปโดยไทย': 'สังคมประชาธิปไตยไทย',

    # เครือข่ายชาวนาแห่งประเทศไทย
    'เครือขายชาวนาแห่งประเทศไทย': 'เครือข่ายชาวนาแห่งประเทศไทย',

    # ใหม่ (พรรคกรีน)
    'กรีน': 'กรีน', 'กริน': 'กรีน',

    # noise / garbage rows
    'รวมคะแนนทั้งสิ้น': None,
}

before_unique = r['name'].nunique()
r['name'] = r['name'].replace(PARTY_MAP)

# ลบแถวที่ map เป็น None (garbage)
garbage = r['name'].isna()
print(f'ลบ garbage name rows: {garbage.sum()}')
r = r[~garbage].copy()

after_unique = r['name'].nunique()
print(f'unique name: {before_unique} → {after_unique}')

### 1.5 แยก 'นอกเขต' ออกเป็น subset

In [ ]:
r_outside = r[r['unit_number'] == -1].copy()
r_inzone  = r[r['unit_number'] != -1].copy()

print(f'ในเขต  : {len(r_inzone):,} rows')
print(f'นอกเขต : {len(r_outside):,} rows')

### 1.6 Cross-validation: score_sum vs valid_ballots

In [ ]:
s_tmp = summary.copy()
s_tmp['type'] = s_tmp['type'].replace('บч', 'บช')
s_tmp = s_tmp[s_tmp['province'] == 'ลำปาง']
ballot_cols = ['total_ballots','valid_ballots','invalid_ballots','no_vote_ballots','remain_ballots']
for col in ballot_cols:
    s_tmp[col] = s_tmp[col].replace(-1, np.nan)
s_tmp = s_tmp.dropna(subset=['total_ballots'])
s_in = s_tmp[s_tmp['unit_number'] != -1]

JOIN = ['type','district','sub-district','unit_number']
score_sum = r_inzone.groupby(JOIN)['score'].sum().reset_index()
score_sum.columns = JOIN + ['score_sum']

cv = score_sum.merge(s_in[JOIN + ['valid_ballots']], on=JOIN, how='inner')
cv['diff'] = cv['score_sum'] - cv['valid_ballots']

outliers = cv[cv['diff'] > 500].sort_values('diff', ascending=False)
print(f'หน่วยที่ score_sum เกิน valid_ballots >500: {len(outliers)} units')
print(outliers[JOIN + ['score_sum','valid_ballots','diff']].to_string())

In [ ]:
# ลบ outlier units ที่ diff > 500 ออกจาก r_inzone (OCR ยังหลุดอยู่)
bad_units = outliers[['type','district','sub-district','unit_number']].drop_duplicates()
bad_key = bad_units.apply(tuple, axis=1).tolist()
r_key = r_inzone[['type','district','sub-district','unit_number']].apply(tuple, axis=1)

before = len(r_inzone)
r_inzone = r_inzone[~r_key.isin(bad_key)].copy()
r_outside_orig = r[r['unit_number'] == -1].copy()
r = pd.concat([r_inzone, r_outside_orig], ignore_index=True)

print(f'ลบ outlier units: {before} → {len(r_inzone)} rows ในเขต (ลบ {before - len(r_inzone)} rows)')
print(f'r ทั้งหมด: {len(r):,} rows')

### 1.7 ตรวจสอบผลลัพธ์

In [ ]:
print('dtype score:', r['score'].dtype)
print('\nสถิติ score (ทุก row ที่เหลือ):')
print(r['score'].describe())
print('\nTop 15 คะแนนรวม (ในเขต, บช):')
top = r_inzone[r_inzone['type']=='บช'].groupby('name')['score'].sum().sort_values(ascending=False).head(15)
print(top.to_string())

### 1.8 Export

In [ ]:
r.to_csv(OUT / 'master_results_cleaned.csv', index=False, encoding='utf-8-sig')
print('Saved master_results_cleaned.csv:', r.shape)

---
## 2. Clean master_summary.csv

### 2.1 แก้ type และ province noise

In [ ]:
s = summary.copy()

s['type'] = s['type'].replace('บч', 'บช')

noise_prov = s[s['province'] != 'ลำปาง']['province'].value_counts()
if len(noise_prov) > 0:
    print('province noise:', noise_prov.to_dict())
s = s[s['province'] == 'ลำปาง'].copy()

print('type unique:', s['type'].unique())
print('province unique:', s['province'].unique())

### 2.2 จัดการค่า -1 (Missing Value)

In [ ]:
ballot_cols = ['total_ballots','valid_ballots','invalid_ballots','no_vote_ballots','remain_ballots']

mask_neg = (s[ballot_cols] == -1).any(axis=1)
print(f'แถวที่มีค่า -1: {mask_neg.sum()} rows')
print(s[mask_neg][['type','district','sub-district','unit_number'] + ballot_cols])

for col in ballot_cols:
    s[col] = s[col].replace(-1, np.nan)

before = len(s)
s = s.dropna(subset=['total_ballots']).copy()
print(f'\nDrop แถว total_ballots=NaN: {before} → {len(s)} rows (ลบ {before - len(s)} rows)')

### 2.3 แยก 'นอกเขต' และตรวจสอบ

In [ ]:
s_outside = s[s['unit_number'] == -1].copy()
s_inzone  = s[s['unit_number'] != -1].copy()

print(f'ในเขต  : {len(s_inzone)} rows')
print(f'นอกเขต : {len(s_outside)} rows')
print('\nสถิติหลัง clean:')
print(s[ballot_cols].describe())

### 2.4 Export

In [ ]:
s.to_csv(OUT / 'master_summary_cleaned.csv', index=False, encoding='utf-8-sig')
print('Saved master_summary_cleaned.csv:', s.shape)

---
## 3. Clean coords (output_with_coordinates.csv)

In [ ]:
c = coords.copy()
print('columns:', list(c.columns))
print('null count:')
print(c.isnull().sum())

has_coord = c[['latitude','longitude']].notnull().all(axis=1).sum()
print(f'\nมีพิกัด: {has_coord}/{len(c)} ({has_coord/len(c)*100:.1f}%)')

dup_coord = c[c.duplicated(subset=['latitude','longitude'], keep=False)]
print(f'พิกัดซ้ำกัน (หลายหน่วยใช้จุดเดียว): {len(dup_coord)} rows')
print('  → ไม่ลบทิ้ง เพราะปกติสำหรับพิกัดระดับหมู่บ้าน, ใช้ jitter ตอนพล็อตแผนที่')

In [ ]:
c.to_csv(OUT / 'coords_cleaned.csv', index=False, encoding='utf-8-sig')
print('Saved coords_cleaned.csv:', c.shape)

---
## 4. สรุป Cleaning Report

In [ ]:
print('=' * 55)
print('CLEANING REPORT')
print('=' * 55)
print(f'master_results : {len(results):,} → {len(r):,} rows  (ลบ {len(results)-len(r):,} rows)')
print(f'  - แก้ type บч → บช')
print(f'  - ลบ province noise')
print(f'  - ลบ score < 0 (missing) และ score > 99999 (OCR ต่อเลข)')
print(f'  - ปัดทศนิยม OCR → Int64')
print(f'  - Normalize ชื่อพรรค (Entity Resolution)')
print(f'  - ลบ outlier units (score_sum >> valid_ballots)')
print()
print(f'master_summary : {len(summary):,} → {len(s):,} rows  (ลบ {len(summary)-len(s):,} rows)')
print(f'  - แก้ type บч → บช')
print(f'  - แทน -1 ด้วย NaN และ drop แถว total_ballots=NaN')
print()
print(f'coords         : {len(coords):,} → {len(c):,} rows  (ไม่มีการลบ)')
print(f'  - ใช้ output_with_coordinates.csv แทน unit_position / unit_village')
print(f'  - พิกัดซ้ำ {len(dup_coord)} rows คงไว้ (jitter ตอนพล็อต)')
print()
print('Output files:')
for f in sorted(OUT.glob('*.csv')):
    df_tmp = pd.read_csv(f)
    print(f'  cleaned/{f.name} — {df_tmp.shape[0]:,} rows x {df_tmp.shape[1]} cols')

---
## 5. ทดสอบ Join cleaned data

In [ ]:
r_clean = pd.read_csv(OUT / 'master_results_cleaned.csv')
s_clean = pd.read_csv(OUT / 'master_summary_cleaned.csv')
c_clean = pd.read_csv(OUT / 'coords_cleaned.csv')

JOIN = ['district', 'sub-district', 'unit_number']

merged = (
    s_clean[s_clean['unit_number'] != -1]
    .merge(c_clean[JOIN + ['village','place','latitude','longitude']], on=JOIN, how='left')
)

has_coord = merged[['latitude','longitude']].notnull().all(axis=1).sum()
print(f'merged shape : {merged.shape}')
print(f'มีพิกัด      : {has_coord}/{len(merged)} ({has_coord/len(merged)*100:.1f}%)')
merged.head(5)